# LLM primer: two-token self-attention

Companion to HTB Academy Module 290, Section 22. These are **hand-picked, untrained vectors**, not an LLM. This notebook isolates the score → softmax → weighted-value calculation. It does not tokenize real text or train a transformer.

Read Attention(Q,K,V)=softmax(QKᵀ/√dₖ)V as **attention of Q, K, V equals softmax of Q times K-transpose over square root of d sub k, then times V**. Queries compare to keys; normalized comparison weights mix the values.

In [ ]:
import math

queries = [(1.0, 0.0), (0.0, 1.0)]
keys = [(1.0, 0.0), (0.0, 1.0)]
values = [(1.0, 0.0), (0.0, 1.0)]
key_width = len(keys[0])  # d_k = 2.

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

def softmax(scores):
    """Stable normalization of one row of attention scores."""
    largest = max(scores)
    exponentials = [math.exp(score - largest) for score in scores]
    denominator = sum(exponentials)
    return [value / denominator for value in exponentials]

outputs = []
for position, query in enumerate(queries):
    scores = [dot(query, key) / math.sqrt(key_width) for key in keys]
    weights = softmax(scores)  # One weight per key/value position.
    mixed = tuple(sum(weights[j] * values[j][dimension] for j in range(len(values)))
                  for dimension in range(len(values[0])))
    print(f'query {position}: scores={scores}, weights={weights}, mixed value={mixed}')
    assert abs(sum(weights) - 1.0) < 1e-12
    outputs.append(mixed)

assert 0.66 < outputs[0][0] < 0.68
assert 0.32 < outputs[0][1] < 0.34
assert abs(outputs[0][0] - outputs[1][1]) < 1e-12
assert abs(outputs[0][1] - outputs[1][0]) < 1e-12

The first query weighs its own value about 0.67 and the other value about 0.33. The second query gets the symmetric result. **This is bidirectional, unmasked attention for illustration.** A causal next-token model must hide future positions when producing a prediction at an earlier position. Attention weights are information-mixing coefficients, not a guarantee of correct language understanding or a defense against prompt injection.